In [ ]:
!pip -q install kagglehub

In [ ]:
import os, kagglehub

path = kagglehub.dataset_download("loki4514/rice-leaf-diseases-detection")
print("Base path:", path)
print("Top folders:", os.listdir(path))

100%|██████████| 8.03G/8.03G [01:08<00:00, 126MB/s]

Extracting files...


Base path: /root/.cache/kagglehub/datasets/loki4514/rice-leaf-diseases-detection/versions/8
Top folders: ['Rice_Leaf_AUG', 'Rice_Leaf_Diease']


In [ ]:
import os

data_path = os.path.join(path, "Rice_Leaf_Diease", "Rice_Leaf_Diease")

train_path = os.path.join(data_path, "train")
test_path  = os.path.join(data_path, "test")

print("Train:", os.listdir(train_path))
print("Test:", os.listdir(test_path))

Train: ['leaf_scald', 'healthy', 'neck_blast', 'tungro', 'narrow_brown_spot', 'brown_spot', 'sheath_blight', 'rice_hispa', 'leaf_blast', 'bacterial_leaf_blight']
Test: ['leaf_scald', 'healthy', 'narrow_brown_spot', 'Sheath Blight', 'brown_spot', 'Tungro', 'leaf_blast', 'bacterial_leaf_blight', 'Neck_Blast', 'Rice Hispa']


In [ ]:
import os

rename_map = {
    "Sheath Blight": "sheath_blight",
    "Tungro": "tungro",
    "Neck_Blast": "neck_blast",
    "Rice Hispa": "rice_hispa",
}

for old, new in rename_map.items():
    old_path = os.path.join(test_path, old)
    new_path = os.path.join(test_path, new)
    if os.path.exists(old_path) and not os.path.exists(new_path):
        os.rename(old_path, new_path)

print("✅ Fixed Test:", os.listdir(test_path))

✅ Fixed Test: ['leaf_scald', 'healthy', 'neck_blast', 'tungro', 'narrow_brown_spot', 'brown_spot', 'sheath_blight', 'rice_hispa', 'leaf_blast', 'bacterial_leaf_blight']


In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
print("✅ Classes:", class_names)

Found 15023 files belonging to 10 classes.
Using 12019 files for training.
Found 15023 files belonging to 10 classes.
Using 3004 files for validation.
Found 3422 files belonging to 10 classes.
✅ Classes: ['bacterial_leaf_blight', 'brown_spot', 'healthy', 'leaf_blast', 'leaf_scald', 'narrow_brown_spot', 'neck_blast', 'rice_hispa', 'sheath_blight', 'tungro']


In [ ]:
print("✅ Fixed Test:", os.listdir(test_path))

✅ Fixed Test: ['leaf_scald', 'healthy', 'neck_blast', 'tungro', 'narrow_brown_spot', 'brown_spot', 'sheath_blight', 'rice_hispa', 'leaf_blast', 'bacterial_leaf_blight']


In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)
print("✅ Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

Found 15023 files belonging to 10 classes.
Using 12019 files for training.
Found 15023 files belonging to 10 classes.
Using 3004 files for validation.
Found 3422 files belonging to 10 classes.
✅ Classes: ['bacterial_leaf_blight', 'brown_spot', 'healthy', 'leaf_blast', 'leaf_scald', 'narrow_brown_spot', 'neck_blast', 'rice_hispa', 'sheath_blight', 'tungro']


In [ ]:
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")
print("✅ Policy:", mixed_precision.global_policy())

✅ Policy: <DTypePolicy "mixed_float16">


In [ ]:
from tensorflow.keras import layers

data_aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name="data_augmentation")

base = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)
base.trainable = False  # Stage 1

inputs = tf.keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = data_aug(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

# dtype float32 is important when using mixed precision
outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,062,381 (15.50 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_model.keras", monitor="val_accuracy", save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=4, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.2, patience=2, min_lr=1e-6, verbose=1
    )
]

In [ ]:
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=callbacks
)

Epoch 1/12
751/752 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.6392 - loss: 1.1310
Epoch 1: val_accuracy improved from -inf to 0.81858, saving model to best_model.keras
752/752 ━━━━━━━━━━━━━━━━━━━━ 233s 117ms/step - accuracy: 0.6394 - loss: 1.1301 - val_accuracy: 0.8186 - val_loss: 0.5387 - learning_rate: 0.0010
Epoch 2/12
751/752 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.8362 - loss: 0.5089
Epoch 2: val_accuracy improved from 0.81858 to 0.85719, saving model to best_model.keras
752/752 ━━━━━━━━━━━━━━━━━━━━ 181s 78ms/step - accuracy: 0.8363 - loss: 0.5089 - val_accuracy: 0.8572 - val_loss: 0.4057 - learning_rate: 0.0010
Epoch 3/12
751/752 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.8572 - loss: 0.4141
Epoch 3: val_accuracy improved from 0.85719 to 0.88016, saving model to best_model.keras
752/752 ━━━━━━━━━━━━━━━━━━━━ 181s 80ms/step - accuracy: 0.8572 - loss: 0.4140 - val_accuracy: 0.8802 - val_loss: 0.3380 - learning_rate: 0.0010
Epoch 4/12
751/752 ━━━━━━━━━━━━━━━━━━━━ 0s

In [ ]:
model = tf.keras.models.load_model("best_model.keras")

base = model.get_layer("efficientnetb0")
base.trainable = True

# Unfreeze last 30 layers only
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=callbacks
)

Epoch 1/8
751/752 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.7440 - loss: 0.7710
Epoch 1: val_accuracy did not improve from 0.91811
752/752 ━━━━━━━━━━━━━━━━━━━━ 214s 107ms/step - accuracy: 0.7441 - loss: 0.7707 - val_accuracy: 0.8905 - val_loss: 0.3198 - learning_rate: 1.0000e-05
Epoch 2/8
752/752 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.8282 - loss: 0.5097
Epoch 2: val_accuracy did not improve from 0.91811
752/752 ━━━━━━━━━━━━━━━━━━━━ 176s 87ms/step - accuracy: 0.8282 - loss: 0.5096 - val_accuracy: 0.9085 - val_loss: 0.2654 - learning_rate: 1.0000e-05
Epoch 3/8
751/752 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.8569 - loss: 0.4206
Epoch 3: val_accuracy improved from 0.91811 to 0.92077, saving model to best_model.keras
752/752 ━━━━━━━━━━━━━━━━━━━━ 172s 90ms/step - accuracy: 0.8569 - loss: 0.4206 - val_accuracy: 0.9208 - val_loss: 0.2330 - learning_rate: 1.0000e-05
Epoch 4/8
751/752 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.8726 - loss: 0.3627
Epoch 4: val_accurac

In [ ]:
model = tf.keras.models.load_model("best_model.keras")
test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print("✅ Test Accuracy:", test_acc)

214/214 ━━━━━━━━━━━━━━━━━━━━ 66s 284ms/step - accuracy: 0.9458 - loss: 0.1545
✅ Test Accuracy: 0.9453535676002502


In [ ]:
from google.colab import files
uploaded = files.upload()  # choose one rice leaf image


Saving leaf.jpeg to leaf.jpeg


In [ ]:
img_path = list(uploaded.keys())[0]
predict_image(img_path)


1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step
🌾 Disease: brown_spot
📊 Confidence: 0.98


In [ ]:
model.save("final_model.keras")

In [ ]:
import os
print(os.listdir())

['.config', 'leaf.jpeg', 'best_model.keras', 'best_model_v2.keras', 'class_names.json', 'final_model.keras', 'sample_data']


In [ ]:
from google.colab import files
files.download("final_model.keras")
files.download("class_names.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
with open("class_names.json","w") as f:
    json.dump(class_names, f)

In [ ]:
print(os.listdir())

['.config', 'leaf.jpeg', 'best_model.keras', 'best_model_v2.keras', 'class_names.json', 'final_model.keras', 'sample_data']


In [ ]:
from google.colab import files

files.download("final_model.keras")
files.download("class_names.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>